# Step 4: Build Clean US Patent Dataset (includes Continuation Filtering)

## Purpose
Combine the maker–application links (Step 2) with the technology-classified
patent universe (Step 3), restrict to **US patent applications only**, remove
internal PATSTAT maintenance records, and filter out **continuation patents**
(re-filings of previously granted patents).

Continuation patents are removed because they do not represent new inventive
activity — including them would double-count the same innovation.  The
exclusion list is sourced from TLS216 (PATSTAT's continuation links table).

Continuation filtering (TLS216) is done in this step.

## Inputs
| File | Description |
|------|-------------|
| `../step2/major_makers.csv` | 90 major automakers and their application IDs |
| `../step3/patents_cpc_classified.csv` | All 114 M patents with ICE/EV CPC counts |
| `patstat_raw/tls216_part01.csv` | TLS216 continuation patent IDs |

## Outputs
| File | Description |
|------|-------------|
| `us_patents_clean.csv` | US patents for major automakers, continuations removed — read by **Step 5** |
| `unique_inventions_per_maker.xlsx` | Unique patent family counts per firm (inspection aid) |


In [ ]:
import pandas as pd
import gc
from pathlib import Path


In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
MAKERS_FILE  = '../step2/major_makers.csv'
CPC_FILE     = '../step3/patents_cpc_classified.csv'

# TLS216 contains the appln_id of every continuation patent in PATSTAT.
# Raw PATSTAT exports live outside the repository (see SETUP.md, Part 2).
# Default: a folder `patstat_raw/` next to the repository folder; set the
# environment variable PATSTAT_DIR to use another location.
import os
PATSTAT_DIR = Path(os.environ.get('PATSTAT_DIR', '../../../patstat_raw'))
TLS216_FILE  = PATSTAT_DIR / 'tls216_part01.csv'
_missing = [p for p in [TLS216_FILE] if not p.exists()]
if _missing:
    for p in _missing:
        print('MISSING', p)
    raise FileNotFoundError(
        f'{len(_missing)} raw PATSTAT file(s) missing. Put ALL PATSTAT exports into '
        f'{PATSTAT_DIR.resolve()} (default: patstat_raw/ next to the repository folder; '
        'see SETUP.md, Part 2).')

OUTPUT_CSV   = 'us_patents_clean.csv'
OUTPUT_XLSX  = 'unique_inventions_per_maker.xlsx'

# PATSTAT uses application IDs above 900 M for internal bookkeeping records
# that are not genuine patent applications.  These must be excluded.
MAX_VALID_APPLN_ID = 900_000_000


In [ ]:
def load_makers(filepath: str) -> pd.DataFrame:
    """
    Load the curated major-makers table from Step 2.

    Parameters
    ----------
    filepath : str

    Returns
    -------
    pd.DataFrame : (person_id, appln_id, psn_name, person_ctry_code)
    """
    df = pd.read_csv(filepath, usecols=['person_id', 'appln_id',
                                        'person_ctry_code', 'psn_name'])
    print(f'Makers: {df["psn_name"].nunique()} unique firms, {len(df):,} rows')
    return df


In [ ]:
def load_cpc_patents(filepath: str) -> pd.DataFrame:
    """
    Load the technology-classified patent universe from Step 3.

    We use memory-efficient dtypes: int32 for IDs and int16 for counts,
    which are always small non-negative integers (max ~100).

    Parameters
    ----------
    filepath : str

    Returns
    -------
    pd.DataFrame : 114 M rows with appln_id, appln_auth, appln_filing_date,
                   docdb_family_id, count_ICE_codes, count_EV_codes, count_Hybrid_codes
    """
    dtype_map = {
        'appln_id'          : 'int32',
        'docdb_family_id'   : 'int32',
        'count_ICE_codes'   : 'int16',
        'count_EV_codes'    : 'int16',
        'count_Hybrid_codes': 'int16',
    }
    print('Loading CPC-classified patents (114 M rows)...', end=' ', flush=True)
    df = pd.read_csv(filepath, dtype=dtype_map)
    print(f'{len(df):,} rows loaded')
    return df


In [ ]:
def merge_and_filter_us(makers: pd.DataFrame,
                        patents: pd.DataFrame) -> pd.DataFrame:
    """
    Join makers with patents, then restrict to genuine US applications.

    Two filters are applied:
      1. appln_auth == 'US'      — keeps only applications filed at the USPTO.
         US patents are the standard measure of innovation directed toward the
         American market and are widely used in the empirical innovation
         literature (Hall, Jaffe, Trajtenberg 2001).
      2. appln_id <= 900 000 000 — removes internal PATSTAT maintenance records
         that carry artificially high application IDs.

    Parameters
    ----------
    makers  : pd.DataFrame — Step 2 output
    patents : pd.DataFrame — Step 3 output

    Returns
    -------
    pd.DataFrame : US-only, artificial-record-free maker patents
    """
    merged = pd.merge(makers, patents, on='appln_id', how='left')
    print(f'After merge        : {len(merged):,} rows')

    us = merged[
        (merged['appln_auth'] == 'US') &
        (merged['appln_id'] <= MAX_VALID_APPLN_ID)
    ].copy()
    print(f'After US + ID filter: {len(us):,} rows')
    print(f'  Unique appln_id  : {us["appln_id"].nunique():,}')
    print(f'  Unique families  : {us["docdb_family_id"].nunique():,}')
    return us


In [ ]:
def load_continuation_ids(tls216_path: str) -> set:
    """
    Load TLS216 and return all continuation-patent appln_ids as a set.

    Continuation patents are re-filings of previously granted patents —
    they do not represent independent new inventions.  Including them would
    inflate patent counts and bias the trend analysis.

    TLS216 (107 MB) fits in RAM in a single read.

    Parameters
    ----------
    tls216_path : str  — path to tls216_part01.csv

    Returns
    -------
    set : all continuation appln_ids as int64 values
    """
    ids = pd.read_csv(tls216_path, usecols=['appln_id'])['appln_id']
    ids = pd.to_numeric(ids, errors='coerce').dropna().astype('int64')
    excluded = set(ids.tolist())
    print(f'Continuation IDs loaded: {len(excluded):,}')
    return excluded


In [ ]:
def remove_continuations(df: pd.DataFrame, excluded_ids: set) -> pd.DataFrame:
    """
    Drop rows whose appln_id appears in the continuation exclusion set.

    Parameters
    ----------
    df           : pd.DataFrame — US patents before continuation removal
    excluded_ids : set — from load_continuation_ids()

    Returns
    -------
    pd.DataFrame : patents with continuations removed
    """
    before = len(df)
    keep   = ~df['appln_id'].astype('int64').isin(excluded_ids)
    after  = keep.sum()
    print(f'Continuation patents removed: {before - after:,} ({(before - after) / before * 100:.2f}%)')
    print(f'Rows remaining             : {after:,}')
    return df[keep].copy()


def compute_unique_inventions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count unique patent families (docdb_family_id) per firm as a proxy for
    distinct inventions.  Multiple applications belonging to the same family
    are treated as one invention.

    Parameters
    ----------
    df : pd.DataFrame — clean US patents

    Returns
    -------
    pd.DataFrame : (psn_name, unique_invention_count) sorted descending
    """
    return (
        df.groupby('psn_name')['docdb_family_id']
        .nunique()
        .reset_index(name='unique_invention_count')
        .sort_values('unique_invention_count', ascending=False)
    )


In [ ]:
def main() -> None:
    """
    Full Step 4 pipeline:
      1. Load maker–application links (Step 2).
      2. Load technology-classified patents (Step 3).
      3. Merge, filter to US, remove artificial records.
      4. Load continuation IDs from TLS216 and drop them.
      5. Save clean dataset and unique-invention summary.
    """
    print('=== Step 4: Build Clean US Patent Dataset ===')

    # Load inputs
    makers  = load_makers(MAKERS_FILE)
    patents = load_cpc_patents(CPC_FILE)

    # Merge and filter
    print('\nMerging and filtering to US...')
    us = merge_and_filter_us(makers, patents)
    del makers, patents
    gc.collect()

    # Remove continuation patents
    print('\nLoading continuation-patent exclusion list from TLS216...')
    cont_ids = load_continuation_ids(TLS216_FILE)

    us_clean = remove_continuations(us, cont_ids)
    del us, cont_ids
    gc.collect()

    # Save outputs
    inventions = compute_unique_inventions(us_clean)
    inventions.to_excel(OUTPUT_XLSX, index=False)
    print(f'Saved → {OUTPUT_XLSX}')

    us_clean.to_csv(OUTPUT_CSV, index=False)
    print(f'\n✓ Saved → {OUTPUT_CSV}  ({len(us_clean):,} rows)')


main()
